# 10 Predict ranking

Predicts `ranking.parquet` (January and July 2026, the daily leaderboard) with the final models in `model/final`,
written by `09_corrector`. Nothing is trained here.

In [7]:
VERSION = 55          # number of the daily-leaderboard submission
TEAM = "talented-jelly"
UPLOAD = True        # True copies the file to the team bucket (needs the challenge keys)

In [8]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "model/runner.py").is_file())
sys.path.insert(0, str(ROOT / "model"))
import runner
from runner import TARGET

MODELS = ROOT / "model/final"
json.loads((MODELS / "manifest.json").read_text())

{'A': {'run_id': '2a97c58055c641429638c1f74fa74751',
  'run_name': '291_BaseA_CatBoost'},
 'B': {'run_id': 'c492e5c19f3a4e6cac86cf956a48d68a',
  'run_name': '292_BaseB_CatBoost'},
 'C': {'run_id': '0645226cdf6d4b3eadbe7dc06e028ffa',
  'run_name': '294_Classifier_CatBoost'},
 'corrector': {'run_id': 'a305f283e7b144f6aef1e085d84bf44f',
  'run_name': '295_Corrector'}}

## Prediction

Base models, classifier, corrector, mixture, rule and LOBT window (`runner.predict`).

In [9]:
frame = pd.read_parquet(ROOT / "data/03_features/ranking.parquet")
frame = frame.loc[frame["PHASE_mvt"].eq("DEP")].reset_index(drop=True)
frame = runner.predict(frame, MODELS)
frame["month"] = pd.to_datetime(frame["MVT_TIME_UTC_mvt"], utc=True).dt.month
print(f"{len(frame):,} departures, months {sorted(frame['month'].unique().tolist())}")
frame[["base_prediction", "base_normal_prediction", "p_artifact", "prediction"]].describe().round(2)

344,841 departures, months [1, 7]


,base_prediction,base_normal_prediction,p_artifact,prediction
count,344841.00,344841.00,344841.00,344841.00
mean,1041.91,1022.82,0.00,1040.32
std,627.75,389.35,0.04,668.06
min,0.00,0.00,0.00,0.00
25%,769.27,766.50,0.00,762.30
50%,961.57,957.13,0.00,954.01
75%,1212.41,1201.29,0.00,1203.71
max,94780.09,4332.72,1.00,110917.93


In [10]:
frame.groupby(["month", "ADEP_mvt"])["prediction"].median().unstack(0).round(0)

month,1,7
ADEP_mvt,,
EDDF,905.0,863.0
EDDM,844.0,758.0
EGLL,1283.0,1342.0
EHAM,782.0,820.0
LEBL,833.0,940.0
LEMD,978.0,1008.0
LFPG,925.0,915.0
LIRF,971.0,1130.0
LSZH,770.0,706.0


## Submission file

IDs and order of the challenge template `submitting.parquet`.

In [11]:
template = pd.read_parquet(ROOT / "data/01_raw/submitting.parquet")
assert template["MVT_ID_mvt"].is_unique and frame["MVT_ID_mvt"].is_unique, "duplicate flight IDs"
assert set(template["MVT_ID_mvt"]) == set(frame["MVT_ID_mvt"]), "predicted departures differ from the template"
submission = template[["MVT_ID_mvt"]].copy()
submission[TARGET] = submission["MVT_ID_mvt"].map(frame.set_index("MVT_ID_mvt")["prediction"])
assert submission[TARGET].notna().all() and np.isfinite(submission[TARGET]).all(), "missing predictions"
assert submission["MVT_ID_mvt"].equals(template["MVT_ID_mvt"]) and list(submission.columns) == list(template.columns)

path = ROOT / "submissions" / f"{TEAM}_v{VERSION}.parquet"
path.parent.mkdir(exist_ok=True)
submission.to_parquet(path, index=False)
print(f"{path.relative_to(ROOT)}: {len(submission):,} rows, same flights and order as submitting.parquet")
submission[TARGET].describe().round(1)

submissions/talented-jelly_v55.parquet: 344,841 rows, same flights and order as submitting.parquet


count    344841.0
mean       1040.3
std         668.1
min           0.0
25%         762.3
50%         954.0
75%        1203.7
max      110917.9
Name: TAXITIME_SEC_mvt, dtype: float64

## Upload and score

After an upload the organisers score the file against the January/July labels and write
`<file>_result.json` into the team bucket, usually within a minute.

In [12]:
def leaderboard_score(s3, bucket, name, wait_sec=600):
    """Waits for the organisers' result file of an uploaded submission and returns its content (None after wait_sec)."""
    from minio.error import S3Error
    for _ in range(wait_sec // 10):
        try:
            response = s3.get_object(bucket, f"{name}_result.json")
            return json.loads(response.read())
        except S3Error:
            time.sleep(10)
    return None


if UPLOAD:
    import time
    from pyopensky.s3 import S3Client
    s3, bucket = S3Client().s3client, f"prc-2026-{TEAM}"
    s3.fput_object(bucket, path.name, str(path))
    print("uploaded", path.name)
    result = leaderboard_score(s3, bucket, path.name)
    print("no result yet" if result is None else f"{result['status']}: RMSE {result['score']:.2f} on {result['used_pairs']:,} departures")

uploaded talented-jelly_v55.parquet
Succeeded: RMSE 268.68 on 344,841 departures
